# 🃏 TFG — Royal FLush: Introducción a SPADE y Agentes (v2)

**Entorno requerido:** kernel `TFG Royal FLush (Python 3.11)` — entorno `C:\tfg_env311`

---

## 🗺️ Contenido

| Parte | Descripción | ¿Necesita PyJabber? |
|---|---|---|
| 1 | `asyncio` y concurrencia | ❌ No |
| 2 | Primer agente SPADE | ✅ Sí |
| 3 | CyclicBehaviour — bucle de entrenamiento | ✅ Sí |
| 4 | Comunicación entre dos agentes | ✅ Sí |
| 5 | FSM — Máquina de Estados (PACoL) | ✅ Sí |
| 6 | Mapa del código de Royal FLush | ❌ No |

> ⚠️ **Antes de las partes 2-5:** abre una terminal y ejecuta `pyjabber` en el entorno activado.
> Déjala abierta mientras trabajas.

---
# PARTE 1 — La base de todo: `asyncio` 🔄

SPADE usa `asyncio` para ejecutar múltiples agentes **concurrentemente** en un solo hilo.
La clave es `await`: en lugar de bloquear esperando, cede el control para que otros agentes trabajen.

| Concepto | Significado |
|---|---|
| `async def` | Define una corrutina (función que puede pausarse) |
| `await` | Pausa y cede el control al event loop |
| `asyncio.sleep(n)` | Espera N segundos SIN bloquear |
| `asyncio.gather()` | Ejecuta varias corrutinas a la vez |

In [1]:
import asyncio
import time

# ── EJEMPLO 1: bloqueante vs async ──────────────────────────────────────────

async def agente_bloqueante(nombre, segundos):
    print(f"[{nombre}] Inicio")
    time.sleep(segundos)          # ❌ Bloquea TODO el programa
    print(f"[{nombre}] Fin (bloqueante)")

async def agente_async(nombre, segundos):
    print(f"[{nombre}] Inicio")
    await asyncio.sleep(segundos) # ✅ Pausa y deja actuar a otros
    print(f"[{nombre}] Fin (async)")

# Versión bloqueante: 2+2 = ~4 segundos
print("=== BLOQUEANTE ===")
t = time.time()
await agente_bloqueante("A", 2)
await agente_bloqueante("B", 2)
print(f"Tiempo: {time.time()-t:.1f}s\n")

# Versión async: ambos a la vez = ~2 segundos
print("=== ASYNC ===")
t = time.time()
await asyncio.gather(
    agente_async("A", 2),
    agente_async("B", 2),
)
print(f"Tiempo: {time.time()-t:.1f}s")

=== BLOQUEANTE ===
[A] Inicio
[A] Fin (bloqueante)
[B] Inicio
[B] Fin (bloqueante)
Tiempo: 4.0s

=== ASYNC ===
[A] Inicio
[B] Inicio
[A] Fin (async)
[B] Fin (async)
Tiempo: 2.0s


In [2]:
import asyncio

# ── EJEMPLO 2: asyncio.Queue — la herramienta central de PACoL ───────────────
#
# En Royal FLush, el LayerReceiverBehaviour mete modelos en una Queue.
# El ConsensusState los consume cuando llega su turno.
# Aquí lo simulamos sin SPADE.

async def productor(cola: asyncio.Queue, nombre: str):
    """Simula un agente enviando sus pesos."""
    for ronda in range(1, 4):
        modelo = {"fc1.weight": round(ronda * 0.1, 2), "fc2.weight": round(ronda * 0.2, 2)}
        print(f"[{nombre}] → Enviando ronda {ronda}: {modelo}")
        await cola.put(modelo)
        await asyncio.sleep(0.5)

async def consumidor(cola: asyncio.Queue, nombre: str):
    """Simula el ConsensusState procesando modelos recibidos."""
    acumulados = []
    while True:
        try:
            modelo = await asyncio.wait_for(cola.get(), timeout=1.0)
            acumulados.append(modelo)
            print(f"[{nombre}] ← Recibido: {modelo}")
            if len(acumulados) >= 2:
                # Consenso: promedio de fc1.weight
                vals = [m["fc1.weight"] for m in acumulados]
                print(f"[{nombre}] 🤝 Consenso fc1.weight = {sum(vals)/len(vals):.3f}")
                acumulados.clear()
        except asyncio.TimeoutError:
            print(f"[{nombre}] Cola vacía. Fin.")
            break

cola = asyncio.Queue()
await asyncio.gather(
    productor(cola, "Agente-A"),
    consumidor(cola, "Agente-B"),
)
# → Fíjate que productor y consumidor avanzan a la vez (paralelo)

[Agente-A] → Enviando ronda 1: {'fc1.weight': 0.1, 'fc2.weight': 0.2}
[Agente-B] ← Recibido: {'fc1.weight': 0.1, 'fc2.weight': 0.2}
[Agente-A] → Enviando ronda 2: {'fc1.weight': 0.2, 'fc2.weight': 0.4}
[Agente-B] ← Recibido: {'fc1.weight': 0.2, 'fc2.weight': 0.4}
[Agente-B] 🤝 Consenso fc1.weight = 0.150
[Agente-A] → Enviando ronda 3: {'fc1.weight': 0.3, 'fc2.weight': 0.6}
[Agente-B] ← Recibido: {'fc1.weight': 0.3, 'fc2.weight': 0.6}
[Agente-B] Cola vacía. Fin.


[None, None]

---
# PARTE 2 — Primer agente SPADE 🤖

> ✅ Asegúrate de que PyJabber está corriendo antes de ejecutar esta celda.

Un agente SPADE tiene:
- **JID**: identificador único (`nombre@servidor`)
- **setup()**: método async que se ejecuta al arrancar, donde se añaden los Behaviours
- **Behaviours**: las tareas que ejecuta el agente

**Patrón correcto para esperar en SPADE 4.x** (no existe `wait_until_finished()`):
```python
while agente.is_alive():
    await asyncio.sleep(1)
```

In [3]:
import asyncio
from spade.agent import Agent
from spade.behaviour import OneShotBehaviour

class SaludoBehaviour(OneShotBehaviour):
    """OneShotBehaviour: se ejecuta UNA SOLA VEZ y termina."""
    async def run(self):
        print(f"[{self.agent.jid}] 👋 ¡Hola! Soy un agente SPADE.")
        print(f"[{self.agent.jid}] Mi JID es: {self.agent.jid}")
        # El agente se para a sí mismo cuando termina el behaviour
        await self.agent.stop()

class PrimerAgente(Agent):
    async def setup(self):
        print(f"[{self.jid}] ⚙️  Configurando...")
        self.add_behaviour(SaludoBehaviour())

async def ejecutar_primer_agente():
    agente = PrimerAgente("agente1@localhost", "password123")
    print("🚀 Iniciando agente...")
    await agente.start(auto_register=True)

    # ✅ Patrón correcto SPADE 4.x — esperar hasta que el agente pare
    while agente.is_alive():  #await agente.wait_until_finished()
        await asyncio.sleep(1)

    print("🏁 Agente finalizado.")

await ejecutar_primer_agente()

🚀 Iniciando agente...
[agente1@localhost] ⚙️  Configurando...
[agente1@localhost] 👋 ¡Hola! Soy un agente SPADE.
[agente1@localhost] Mi JID es: agente1@localhost
🏁 Agente finalizado.


---
# PARTE 3 — CyclicBehaviour: el bucle de entrenamiento 🔁

`CyclicBehaviour` se ejecuta en bucle indefinido. Es el comportamiento principal de los **Node agents** en Royal FLush: entrenan, comunican, hacen consenso... y repiten.

In [4]:
import asyncio
from spade.agent import Agent
from spade.behaviour import CyclicBehaviour

class BucleEntrenamiento(CyclicBehaviour):
    """
    Simula el ciclo Train → Communicate → Consensus de Royal FLush.
    En RF este ciclo lo gestiona la FSMBehaviour (Parte 5 del notebook).
    """
    def __init__(self, max_rondas=3):
        super().__init__()
        self.ronda = 0
        self.max_rondas = max_rondas
        self.pesos = {"fc1": 0.50, "fc2": 0.30}  # En RF: state_dict() de PyTorch

    async def run(self):
        """Este método se llama en cada iteración del bucle."""
        if self.ronda >= self.max_rondas:
            print(f"[{self.agent.jid.node}] 🏁 Fin ({self.max_rondas} rondas completadas).")
            await self.agent.stop()
            return

        self.ronda += 1
        print(f"\n[{self.agent.jid.node}] ══ RONDA {self.ronda}/{self.max_rondas} ══")

        # ── FASE 1: ENTRENAMIENTO ──────────────────────────────────────
        print(f"  [Train] 🧠 Entrenando... ", end="")
        await asyncio.sleep(0.3)   # En RF: forward + backward de PyTorch
        self.pesos = {k: round(v + 0.01 * self.ronda, 4) for k, v in self.pesos.items()}
        print(f"pesos actualizados: {self.pesos}")

        # ── FASE 2: COMUNICACIÓN ───────────────────────────────────────
        print(f"  [Comm]  📤 Enviando pesos a vecino... ", end="")
        await asyncio.sleep(0.1)   # En RF: send() via XMPP
        print("enviados.")

        # ── FASE 3: CONSENSO ───────────────────────────────────────────
        print(f"  [Cons]  🤝 Fusionando modelos... ", end="")
        await asyncio.sleep(0.1)   # En RF: promedio de tensores de la Queue
        vecino_simulado = {k: v - 0.05 for k, v in self.pesos.items()}  # modelo ficticio
        self.pesos = {k: round((self.pesos[k] + vecino_simulado[k]) / 2, 4) for k in self.pesos}
        print(f"pesos tras consenso: {self.pesos}")

class AgenteEntrenamiento(Agent):
    async def setup(self):
        self.add_behaviour(BucleEntrenamiento(max_rondas=3))

async def ejecutar_entrenamiento():
    agente = AgenteEntrenamiento("nodo1@localhost", "pass123")
    await agente.start(auto_register=True)
    while agente.is_alive():
        await asyncio.sleep(1)
    print("\n✅ Ciclo de entrenamiento completado.")

await ejecutar_entrenamiento()


[nodo1] ══ RONDA 1/3 ══
  [Train] 🧠 Entrenando... pesos actualizados: {'fc1': 0.51, 'fc2': 0.31}
  [Comm]  📤 Enviando pesos a vecino... enviados.
  [Cons]  🤝 Fusionando modelos... pesos tras consenso: {'fc1': 0.485, 'fc2': 0.285}

[nodo1] ══ RONDA 2/3 ══
  [Train] 🧠 Entrenando... pesos actualizados: {'fc1': 0.505, 'fc2': 0.305}
  [Comm]  📤 Enviando pesos a vecino... enviados.
  [Cons]  🤝 Fusionando modelos... pesos tras consenso: {'fc1': 0.48, 'fc2': 0.28}

[nodo1] ══ RONDA 3/3 ══
  [Train] 🧠 Entrenando... pesos actualizados: {'fc1': 0.51, 'fc2': 0.31}
  [Comm]  📤 Enviando pesos a vecino... enviados.
  [Cons]  🤝 Fusionando modelos... pesos tras consenso: {'fc1': 0.485, 'fc2': 0.285}
[nodo1] 🏁 Fin (3 rondas completadas).

✅ Ciclo de entrenamiento completado.


---
# PARTE 4 — Comunicación entre agentes 📨

Dos agentes enviándose pesos de modelo por XMPP. Conceptos clave:

- `Message(to, body, metadata)` — construye un mensaje XMPP
- `await self.send(msg)` — lo envía
- `await self.receive(timeout=N)` — espera N segundos a recibir uno
- `Template(metadata={...})` — filtra qué mensajes procesa cada behaviour

**Patrón correcto para el receptor:** bucle corto (timeout=1s) que nunca para por timeout, solo para cuando recibe el mensaje. El control de parada total lo maneja la función principal.

In [5]:
import asyncio
import json
from spade.agent import Agent
from spade.behaviour import CyclicBehaviour, OneShotBehaviour
from spade.message import Message

# ── EMISOR ────────────────────────────────────────────────────────────────────
class EmisorBehaviour(OneShotBehaviour):
    """Simula el CommunicationState de Royal FLush."""
    async def run(self):
        # Espera a que el receptor esté completamente conectado al servidor XMPP
        print("[emisor] ⏳ Esperando a que el receptor se conecte...")
        await asyncio.sleep(5)

        pesos_modelo = {
            "conv1.weight": [0.12, 0.45, -0.23],
            "fc2.weight":   [0.67, -0.11, 0.88]
        }

        # En RF los mensajes llevan metadata rf.conversation para que el Template filtre
        msg = Message(
            to="receptor@localhost",
            body=json.dumps(pesos_modelo),
        )
        msg.set_metadata("rf.conversation", "layers")

        await self.send(msg)
        print(f"[emisor] 📤 Pesos enviados: {list(pesos_modelo.keys())}")
        await self.agent.stop()


# ── RECEPTOR ──────────────────────────────────────────────────────────────────
class ReceptorBehaviour(CyclicBehaviour):
    """
    Simula el LayerReceiverBehaviour de Royal FLush.

    PATRÓN CORRECTO: timeout corto (1s) en un bucle que NO para por timeout.
    Solo para cuando recibe un mensaje. El timeout global lo controla main().
    """
    async def run(self):
        # Comprueba cada 1s si llegó algo — nunca sale por timeout
        msg = await self.receive(timeout=1)

        if msg is None:
            # No llegó nada, sigue esperando (el bucle llama a run() de nuevo)
            return

        # ── Mensaje recibido ──
        pesos_recibidos = json.loads(msg.body)
        print(f"\n[receptor] 📥 Mensaje recibido de: {msg.sender}")
        print(f"[receptor] Capas recibidas: {list(pesos_recibidos.keys())}")

        # Simula el ConsensusState: promedio de pesos
        mis_pesos = {
            "conv1.weight": [0.20, 0.30, -0.10],
            "fc2.weight":   [0.50, 0.10,  0.70]
        }
        consenso = {
            capa: [round((a + b) / 2, 3) for a, b in zip(mis_pesos[capa], recibidos)]
            for capa, recibidos in pesos_recibidos.items()
        }
        print("[receptor] 🤝 Consenso calculado:")
        for capa, pesos in consenso.items():
            print(f"  {capa}: {pesos}")

        await self.agent.stop()


# ── AGENTES ───────────────────────────────────────────────────────────────────
class AgenteEmisor(Agent):
    async def setup(self):
        self.add_behaviour(EmisorBehaviour())

class AgenteReceptor(Agent):
    async def setup(self):
        # Sin Template: acepta cualquier mensaje
        # (en RF usarían Template(metadata={"rf.conversation": "layers"}))
        self.add_behaviour(ReceptorBehaviour())


# ── FUNCIÓN PRINCIPAL ─────────────────────────────────────────────────────────
async def main_comunicacion():
    receptor = AgenteReceptor("receptor@localhost", "pass2")
    emisor   = AgenteEmisor("emisor@localhost",    "pass1")

    # El receptor arranca primero para estar listo cuando llegue el mensaje
    print("[main] Arrancando receptor...")
    await receptor.start(auto_register=True)
    await asyncio.sleep(1)  # Pequeña pausa para que se conecte

    print("[main] Arrancando emisor...")
    await emisor.start(auto_register=True)

    # Espera con timeout global de 30 segundos
    print("[main] Esperando a que terminen ambos agentes (máx. 30s)...")
    for _ in range(30):
        await asyncio.sleep(1)
        if not emisor.is_alive() and not receptor.is_alive():
            break

    # Limpieza: para cualquier agente que quede vivo
    if emisor.is_alive():   await emisor.stop()
    if receptor.is_alive(): await receptor.stop()

    print("\n✅ Experimento de comunicación completado.")

await main_comunicacion()

[main] Arrancando receptor...
[main] Arrancando emisor...
[main] Esperando a que terminen ambos agentes (máx. 30s)...
[emisor] ⏳ Esperando a que el receptor se conecte...
[emisor] 📤 Pesos enviados: ['conv1.weight', 'fc2.weight']

[receptor] 📥 Mensaje recibido de: emisor@localhost
[receptor] Capas recibidas: ['conv1.weight', 'fc2.weight']
[receptor] 🤝 Consenso calculado:
  conv1.weight: [0.16, 0.375, -0.165]
  fc2.weight: [0.585, -0.005, 0.79]

✅ Experimento de comunicación completado.


---
# PARTE 5 — FSM: la Máquina de Estados de PACoL 🔀

La `FSMBehaviour` organiza al agente en **estados** con **transiciones** definidas.
Este es el corazón de Royal FLush (`behaviour/ppremiofl/fsm.py`):

```
TrainState ──→ CommunicationState ──→ ConsensusState ──┐
    ↑                  ↑                    │           │
    └──────────────────┴────────────────────┘           │
                                                        │
               (LayerReceiverBehaviour corre en paralelo, siempre escuchando)
```

In [6]:
import asyncio
import random
from spade.agent import Agent
from spade.behaviour import FSMBehaviour, State

# ── Nombres de estados (exactamente como en RF) ───────────────────────────────
TRAIN      = "train"
COMM       = "communication"
CONSENSUS  = "consensus"


class TrainState(State):
    """
    Equivale a royalflush/behaviour/ppremiofl/train.py → TrainState
    En RF: llama a model_manager.train() con PyTorch.
    """
    async def run(self):
        ag = self.agent

        # Comprueba si alcanzamos el límite de rondas (igual que en RF)
        if ag.ronda > ag.max_rondas:
            print(f"[{ag.jid.node}] 🏁 Ronda {ag.ronda-1}/{ag.max_rondas} — fin del experimento.")
            await ag.stop()
            return

        print(f"\n[{ag.jid.node}] ══ RONDA {ag.ronda}/{ag.max_rondas} ══")
        print(f"  [Train] 🧠 Entrenando modelo...", end=" ")
        await asyncio.sleep(0.3)  # En RF: forward + backward de PyTorch

        # Simula mejora de accuracy
        ag.accuracy = min(0.99, ag.accuracy + random.uniform(0.02, 0.06))
        ag.pesos["fc1"] = round(ag.pesos["fc1"] + 0.01 * ag.ronda, 4)
        print(f"accuracy={ag.accuracy:.3f}, fc1={ag.pesos['fc1']}")

        ag.ronda += 1
        ag.iter_consenso = 0  # Reinicia contador de consenso para esta ronda
        self.set_next_state(COMM)


class CommunicationState(State):
    """
    Equivale a royalflush/behaviour/ppremiofl/communication.py → ParallelCommunicationState
    En RF: selecciona vecino aleatorio y le envía capas del modelo via XMPP.
    """
    async def run(self):
        ag = self.agent
        print(f"  [Comm]  📤 Enviando pesos a vecino aleatorio...", end=" ")
        await asyncio.sleep(0.1)  # En RF: send() via XMPP
        print("enviado.")

        # Decide si hacer más iteraciones de consenso o volver a Train
        if ag.iter_consenso < ag.max_iter_consenso:
            ag.iter_consenso += 1
            print(f"  [Comm]  → Consenso (iteración {ag.iter_consenso}/{ag.max_iter_consenso})")
            self.set_next_state(CONSENSUS)
        else:
            print(f"  [Comm]  → Volviendo a Train")
            self.set_next_state(TRAIN)


class ConsensusState(State):
    """
    Equivale a royalflush/behaviour/ppremiofl/consensus.py → ParallelConsensusState
    En RF: vacía la Queue de modelos recibidos y hace promedio de pesos.
    """
    async def run(self):
        ag = self.agent
        print(f"  [Cons]  🤝 Aplicando consenso...", end=" ")
        await asyncio.sleep(0.1)  # En RF: promedio de tensores

        # Simula promedio de pesos con el vecino (en RF: state_dict() de PyTorch)
        peso_vecino = ag.pesos["fc1"] + random.uniform(-0.03, 0.03)
        ag.pesos["fc1"] = round((ag.pesos["fc1"] + peso_vecino) / 2, 4)
        print(f"fc1 tras consenso = {ag.pesos['fc1']}")

        # En RF esto lo decide ConsensusState.on_end() comparando iteraciones
        if ag.iter_consenso < ag.max_iter_consenso:
            self.set_next_state(COMM)
        else:
            self.set_next_state(TRAIN)


class PacolFSM(FSMBehaviour):
    """
    Equivale a royalflush/behaviour/ppremiofl/fsm.py → ParallelPremioFsmBehaviour
    """
    def setup(self):
        self.add_state(TRAIN,     TrainState(),         initial=True)
        self.add_state(COMM,      CommunicationState())
        self.add_state(CONSENSUS, ConsensusState())

        # Transiciones permitidas (igual que en RF)
        self.add_transition(TRAIN,     COMM)
        self.add_transition(COMM,      TRAIN)
        self.add_transition(COMM,      CONSENSUS)
        self.add_transition(CONSENSUS, COMM)
        self.add_transition(CONSENSUS, TRAIN)

    async def on_start(self):
        print(f"[{self.agent.jid.node}] 🚀 FSM PACoL iniciada.")

    async def on_end(self):
        print(f"[{self.agent.jid.node}] 🏁 FSM PACoL terminada.")
        print(f"[{self.agent.jid.node}] Pesos finales: {self.agent.pesos}")
        print(f"[{self.agent.jid.node}] Accuracy final: {self.agent.accuracy:.3f}")


class AgentePACoL(Agent):
    """Simula un Node agent de Royal FLush ejecutando PACoL."""
    async def setup(self):
        # Estado interno del agente
        # En RF: estos atributos los gestionan ModelManager y ConsensusManager
        self.ronda = 1
        self.max_rondas = 3
        self.iter_consenso = 0
        self.max_iter_consenso = 2   # En RF: ⌊n_agentes/2⌋
        self.pesos = {"fc1": 0.50, "fc2": 0.30}
        self.accuracy = 0.10

        self.add_behaviour(PacolFSM())


async def main_fsm():
    agente = AgentePACoL("nodo1@localhost", "pass123")
    await agente.start(auto_register=True)
    while agente.is_alive():
        await asyncio.sleep(1)
    print("\n✅ Simulación PACoL completada.")

await main_fsm()

[nodo1] 🚀 FSM PACoL iniciada.

[nodo1] ══ RONDA 1/3 ══
  [Train] 🧠 Entrenando modelo... accuracy=0.129, fc1=0.51
  [Comm]  📤 Enviando pesos a vecino aleatorio... enviado.
  [Comm]  → Consenso (iteración 1/2)
  [Cons]  🤝 Aplicando consenso... fc1 tras consenso = 0.4985
  [Comm]  📤 Enviando pesos a vecino aleatorio... enviado.
  [Comm]  → Consenso (iteración 2/2)
  [Cons]  🤝 Aplicando consenso... fc1 tras consenso = 0.4916

[nodo1] ══ RONDA 2/3 ══
  [Train] 🧠 Entrenando modelo... accuracy=0.185, fc1=0.5116
  [Comm]  📤 Enviando pesos a vecino aleatorio... enviado.
  [Comm]  → Consenso (iteración 1/2)
  [Cons]  🤝 Aplicando consenso... fc1 tras consenso = 0.5217
  [Comm]  📤 Enviando pesos a vecino aleatorio... enviado.
  [Comm]  → Consenso (iteración 2/2)
  [Cons]  🤝 Aplicando consenso... fc1 tras consenso = 0.5363

[nodo1] ══ RONDA 3/3 ══
  [Train] 🧠 Entrenando modelo... accuracy=0.216, fc1=0.5663
  [Comm]  📤 Enviando pesos a vecino aleatorio... enviado.
  [Comm]  → Consenso (iteración 1/2

---
# PARTE 6 — Mapa del código de Royal FLush 🗺️

Esta sección no necesita ni PyJabber ni SPADE — solo explora el código.

In [ ]:
# Mapa visual de la arquitectura
print("""
╔════════════════════════════════════════════════════════════════════╗
║               ARQUITECTURA DE ROYAL FLUSH                          ║
╠════════════════════════════════════════════════════════════════════╣
║                                                                     ║
║  royalflush/                                                        ║
║  │                                                                  ║
║  ├── agent/                       ← Clases de agentes               ║
║  │   ├── base.py                  AgentBase, AgentNode, FlAgent     ║
║  │   ├── coordinator.py           Sincroniza el inicio              ║
║  │   ├── observer.py              Solo registra métricas (CSV)      ║
║  │   └── algorithms/                                                ║
║  │       ├── acol.py              AcolAgent — vecino aleatorio      ║
║  │       └── macofl.py            MaCoFL — vecino por similitud     ║
║  │                                                                  ║
║  ├── behaviour/                   ← Comportamientos (Behaviours)    ║
║  │   └── ppremiofl/                                                 ║
║  │       ├── fsm.py               ParallelPremioFsmBehaviour        ║
║  │       │                        └─ Parte 5 de este notebook       ║
║  │       ├── train.py             TrainState         (Estado 1)     ║
║  │       ├── communication.py     CommunicationState (Estado 2)     ║
║  │       ├── consensus.py         ConsensusState     (Estado 3)     ║
║  │       ├── layer_receiver.py    CyclicBehaviour — siempre activo  ║
║  │       │                        └─ Mete modelos en Queue          ║
║  │       └── similarity_receiver  CyclicBehaviour — vectores        ║
║  │                                                                  ║
║  ├── core/                        ← Lógica de ML                   ║
║  │   ├── models/cnn.py            CNN5 (modelo del paper)          ║
║  │   ├── models/mlp.py            MLP sencillo                     ║
║  │   ├── datasets/cifar.py        CIFAR-10 IID / non-IID           ║
║  │   ├── consensus.py             Operación de promedio de pesos   ║
║  │   └── consensus_manager.py     Gestiona la Queue de consenso    ║
║  │                                                                  ║
║  ├── config/                      ← Lee e01.yml / e02.yml          ║
║  ├── log/                         ← CSV logs (train, inference...)  ║
║  ├── wire/                        ← Serializa mensajes XMPP        ║
║  └── runner/                                                        ║
║      └── dispatcher.py            ← ⚠️  TIENE TODOs — trabajo TFG  ║
╚════════════════════════════════════════════════════════════════════╝
""")

In [1]:
import sys, os

# ── Ajusta esta ruta al ZIP descomprimido ─────────────────────────────────────
RUTA_RF = r"C:\Users\asus  fx506\Desktop\Universidad\TFG\Royal"   # ← CAMBIA ESTO

if os.path.exists(RUTA_RF):
    sys.path.insert(0, RUTA_RF)
    print(f"✅ Royal FLush encontrado en: {RUTA_RF}")
else:
    print(f"⚠️  No encontrado en: {RUTA_RF}")
    print("   Ajusta la variable RUTA_RF")

✅ Royal FLush encontrado en: C:\Users\asus  fx506\Desktop\Universidad\TFG\Royal


In [2]:
# ── Inspecciona el dispatcher.py — aquí están los TODOs del TFG ──────────────
import inspect
from royalflush.runner.dispatcher import RunnerDispatcher

print("=== _run_local() — el método principal a completar ===")
print(inspect.getsource(RunnerDispatcher._run_local))
print()
print("=== __run_local() — la corrutina async ===")
print(inspect.getsource(RunnerDispatcher._RunnerDispatcher__run_local))

=== _run_local() — el método principal a completar ===
    def _run_local(self) -> int:
        if self.config is None:
            self.logger.error("'local' mode requires a config.")
            return 1
        self.logger.info("Local mode selected. Launching all agents...")
        resolved_config = ConfigResolver.resolve(self.config)
        spade.run(self.__run_local(experiment_config=resolved_config))
        # TODO: launch CoordinatorAgent, ObserverAgents, LauncherAgent, NodeAgents
        return 0


=== __run_local() — la corrutina async ===
    async def __run_local(self, experiment_config: ResolvedConfig) -> None:
        deployed_agents: List[AgentBase] = []
        try:
            coordinator = self.__build_coordinator(experiment_config=experiment_config)
            deployed_agents.append(coordinator)
            await coordinator.start()
            await asyncio.sleep(0.2)

            # launchers = self.__build_launchers(experiment_config=experiment_config)
          

In [5]:
# ── Inspecciona el modelo CNN5 del paper ─────────────────────────────────────
import torch
from royalflush.core.models.cnn import CNN5

modelo = CNN5(input_dim=(3, 32, 32), out_classes=10)
print("Arquitectura CNN5:")
print(modelo)

total = sum(p.numel() for p in modelo.parameters())
print(f"\nTotal parámetros: {total:,}")

# Forward pass con imagen CIFAR-10 simulada (batch=1, 3 canales, 32x32)
x = torch.randn(1, 3, 32, 32)
with torch.no_grad():
    salida = modelo(x)
print(f"Forma de salida: {salida.shape}  ← (1 imagen, 10 clases CIFAR-10)")

Arquitectura CNN5:
CNN5(
  (conv1): Conv2d(3, 32, kernel_size=(5, 5), stride=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(5, 5), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=1600, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=10, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
)

Total parámetros: 878,538
Forma de salida: torch.Size([1, 10])  ← (1 imagen, 10 clases CIFAR-10)


In [4]:
import inspect
from royalflush.core.models.cnn import CNN5
print(inspect.signature(CNN5.__init__))

(self, input_dim=(3, 32, 32), out_classes=10)


---
# 📝 Ejercicios

**Ejercicio 1 (Fácil):** Modifica `BucleEntrenamiento` (Parte 3) para que imprima la accuracy después de cada ronda. Añade un atributo `self.accuracy = 0.10` y súbelo en cada ciclo.

**Ejercicio 2 (Medio):** En la Parte 4, añade un segundo emisor (`emisor2@localhost`) que también envíe pesos al receptor. Modifica el receptor para que haga consenso con ambos mensajes.

**Ejercicio 3 (Avanzado — conecta con el TFG):** Mira `royalflush/runner/dispatcher.py`. El método `_run_local()` tiene un TODO para lanzar los NodeAgents. Usando `dispatcher.__build_coordinator()` como referencia, implementa `__build_nodes()` que construya los agentes Node a partir de la configuración YAML.

---
# 📚 Resumen rápido

| Concepto aprendido | Dónde lo usa Royal FLush |
|---|---|
| `async def` / `await` | Todos los Behaviours y métodos de agente |
| `asyncio.Queue` | `FlAgent.consensus_transmissions` en `base.py` |
| `Agent` + `setup()` | `AgentBase`, `AgentNode`, `FlAgent` |
| `OneShotBehaviour` | Fases de coordinación inicial |
| `CyclicBehaviour` | `LayerReceiverBehaviour` (siempre escuchando) |
| `FSMBehaviour` | `ParallelPremioFsmBehaviour` |
| `is_alive()` loop | `dispatcher.py` → `while any(a.is_alive() ...)` |
| `Message` + `set_metadata` | Filtrado por `rf.conversation` en `FlAgent` |